%% [markdown]
# EndoSAM - Notebook 2: MedSAM Experiments
All MedSAM based experiments for pelvic organ segmentation.

**Experiments in this notebook:**
1. MedSAM oracle bounding box (zero-shot)
2. MedSAM simulated manual bounding box
3. MedSAM decoder fine-tuning
4. Decoder test evaluation
5. Auto-box pipeline (U-Net boxes -> MedSAM)
6. Auto-box pipeline (Binary boxes -> MedSAM)

**Requirements:**
- MedSAM weights: medsam_vit_b.pth from Zenodo
- segment-anything: pip install git+https://github.com/facebookresearch/segment-anything.git
- For GPU (Colab): cells 1-2 run faster on GPU
- For CPU (Mac): cells 3-6 work on CPU with pre-computed embeddings

## Cell 0: Imports and Config

In [ ]:
import os, json, time, warnings, numpy as np, nibabel as nib, torch
import torch.nn.functional as F
from PIL import Image
from segment_anything import sam_model_registry
warnings.filterwarnings("ignore")

# ===== Data and output paths =====
DATASET_ROOT = "/path/to/UT-EndoMRI"
MEDSAM_WEIGHTS = os.path.expanduser("~/Desktop/EndoSAM_DecoderFT/medsam_vit_b.pth")
SAVE_DIR = os.path.expanduser("~/Desktop/EndoSAM_DecoderFT")
os.makedirs(SAVE_DIR, exist_ok=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Dataset folders
PREFIX_MAP = {}
for item in os.listdir(DATASET_ROOT):
    if os.path.isdir(os.path.join(DATASET_ROOT, item)):
        if item.startswith("D1"): PREFIX_MAP["D1"] = item
        elif item.startswith("D2"): PREFIX_MAP["D2"] = item

# Subject split (same seed as notebook 1)
all_subjects = []
for pf in ["D1", "D2"]:
    fd = PREFIX_MAP.get(pf)
    if fd is None: continue
    pd = os.path.join(DATASET_ROOT, fd)
    for d in sorted(os.listdir(pd)):
        if d.startswith(pf + "-") and os.path.isdir(os.path.join(pd, d)):
            all_subjects.append(d)

idx = np.random.permutation(len(all_subjects))
n_tr = int(0.70 * len(all_subjects))
n_va = int(0.15 * len(all_subjects))
train_ids = set(all_subjects[i] for i in idx[:n_tr])
test_ids = [all_subjects[i] for i in idx[n_tr+n_va:]]
print(f"Device: {device}")
print(f"Subjects: {len(all_subjects)} total, {len(train_ids)} train, {len(test_ids)} test")

## Cell 1: Helper Functions

In [ ]:
def norm_pct(img_2d):
    fg = img_2d[img_2d > 0]
    if len(fg) == 0:
        return np.zeros_like(img_2d, dtype=np.float32)
    p1, p99 = np.percentile(fg, [1, 99])
    return np.clip((img_2d - p1) / (p99 - p1 + 1e-8), 0, 1).astype(np.float32)

def find_matching_image(sdir, sid, mask_shape):
    for f in os.listdir(sdir):
        if not f.endswith(".nii.gz"): continue
        if any(x in f for x in ["_ut", "_ov", "_em", "_cy", "_pat"]): continue
        s = nib.load(os.path.join(sdir, f)).shape
        if s[0] == mask_shape[0] and s[1] == mask_shape[1]:
            return os.path.join(sdir, f)
    return None

def dice(pred, gt):
    p, g = pred.astype(bool), gt.astype(bool)
    if g.sum() == 0 and p.sum() == 0: return 1.0
    if g.sum() == 0 or p.sum() == 0: return 0.0
    return float(2 * (p & g).sum() / (p.sum() + g.sum()))

def load_medsam():
    """Load MedSAM with official Zenodo weights"""
    medsam = sam_model_registry["vit_b"](checkpoint=None).to(device)
    sd = torch.load(MEDSAM_WEIGHTS, map_location="cpu", weights_only=False)
    medsam.load_state_dict(sd)
    del sd
    medsam.eval()
    print(f"MedSAM loaded on {device}")
    return medsam

@torch.no_grad()
def medsam_predict(medsam, img_2d, bbox):
    """Run MedSAM inference with a bounding box prompt"""
    h, w = img_2d.shape
    # Percentile normalize and convert to 3-channel uint8
    fg = img_2d[img_2d > 0]
    if len(fg) == 0:
        return np.zeros((h, w), dtype=np.uint8)
    p1, p99 = np.percentile(fg, [1, 99])
    img_n = np.clip(img_2d, p1, p99)
    img_n = (img_n - p1) / (p99 - p1 + 1e-8) * 255.0
    img_3c = np.stack([img_n] * 3, axis=-1).astype(np.uint8)
    
    # Resize to 1024x1024 as required by ViT-B encoder
    img_1024 = np.array(Image.fromarray(img_3c).resize((1024, 1024)))
    img_t = torch.from_numpy(img_1024).permute(2, 0, 1).unsqueeze(0).float().to(device)
    
    # Scale bbox to 1024x1024
    sx, sy = 1024 / w, 1024 / h
    bbox_t = torch.tensor([[bbox[0]*sx, bbox[1]*sy, bbox[2]*sx, bbox[3]*sy]]).float().to(device)
    
    # Forward pass
    emb = medsam.image_encoder(img_t)
    sp, dn = medsam.prompt_encoder(points=None, boxes=bbox_t, masks=None)
    lr, _ = medsam.mask_decoder(
        image_embeddings=emb,
        image_pe=medsam.prompt_encoder.get_dense_pe(),
        sparse_prompt_embeddings=sp,
        dense_prompt_embeddings=dn,
        multimask_output=False
    )
    
    # Resize prediction back to original size
    mp = torch.sigmoid(lr)
    mp = F.interpolate(mp, (1024, 1024), mode="bilinear", align_corners=False)
    mp = F.interpolate(mp, (h, w), mode="bilinear", align_corners=False)
    return (mp[0, 0].cpu().numpy() > 0.5).astype(np.uint8)

## Cell 2: MedSAM Oracle Bounding Box (Zero-Shot)
Oracle = tightest box around ground truth + margin. Best possible prompt.

In [ ]:
medsam = load_medsam()

SMAP = {"ut": "uterus", "ov": "ovary", "em": "endo_cyst", "cy": "endo_cyst"}
oracle_results = []
t0 = time.time()

for sid in test_ids:
    prefix = sid.split("-")[0]
    sdir = os.path.join(DATASET_ROOT, PREFIX_MAP[prefix], sid)
    if not os.path.isdir(sdir): continue
    
    for sc in ["ut", "ov", "em", "cy"]:
        mask_path = None
        for c in [f"{sid}_{sc}.nii.gz", f"{sid}_{sc}_r3.nii.gz",
                  f"{sid}_{sc}_r2.nii.gz", f"{sid}_{sc}_r1.nii.gz",
                  f"{sid}_{sc}_re3.nii.gz"]:
            fp = os.path.join(sdir, c)
            if os.path.exists(fp):
                mask_path = fp
                break
        if mask_path is None: continue
        
        mask_nib = nib.load(mask_path)
        mdata = mask_nib.get_fdata()
        if mdata.max() > 1 or np.unique(mdata).size > 2: continue
        gt_mask = (mdata > 0.5).astype(np.uint8)
        
        img_path = find_matching_image(sdir, sid, mask_nib.shape)
        if img_path is None: continue
        
        img_vol = nib.load(img_path).get_fdata().astype(np.float32)
        n_sl = min(img_vol.shape[2], gt_mask.shape[2])
        
        for sl in range(n_sl):
            m2d = gt_mask[:, :, sl]
            if m2d.sum() < 10: continue
            if img_vol[:, :, sl].max() == 0: continue
            
            # Oracle bbox: tight box + 5px margin
            ys, xs = np.where(m2d > 0)
            h, w = img_vol.shape[:2]
            bbox = [max(0, int(xs.min()-5)), max(0, int(ys.min()-5)),
                    min(w, int(xs.max()+5)), min(h, int(ys.max()+5))]
            
            pred = medsam_predict(medsam, img_vol[:, :, sl], bbox)
            d = dice(pred, m2d)
            
            oracle_results.append({
                "subject": sid, "slice": int(sl),
                "structure": SMAP[sc], "dice_oracle": round(d, 4)
            })
            
    if len(oracle_results) % 50 == 0:
        print(f"  {len(oracle_results)} slices processed...")

print(f"\nOracle results ({len(oracle_results)} slices, {time.time()-t0:.0f}s):")
for s in ["uterus", "ovary", "endo_cyst"]:
    e = [r["dice_oracle"] for r in oracle_results if r["structure"] == s]
    if e:
        print(f"  {s:12s}: {np.mean(e):.3f} +/- {np.std(e):.3f} ({len(e)} slices)")

with open(os.path.join(SAVE_DIR, "oracle_results.json"), "w") as f:
    json.dump(oracle_results, f, indent=2)

## Cell 3: MedSAM Simulated Manual Bounding Box
Noise formula: sigma = bbox_dim * 0.15 * (1 - inter_rater_dice)
5 noisy boxes per slice, averaged.

In [ ]:
INTER_RATER = {"uterus": 0.605, "ovary": 0.345, "endo_cyst": 0.40}

simulated_results = []
for i, entry in enumerate(oracle_results):
    sid, sl, struct = entry["subject"], entry["slice"], entry["structure"]
    prefix = sid.split("-")[0]
    sdir = os.path.join(DATASET_ROOT, PREFIX_MAP[prefix], sid)
    
    sc = {"uterus": "ut", "ovary": "ov", "endo_cyst": "em"}.get(struct, "em")
    mask_path = None
    for c in [f"{sid}_{sc}.nii.gz", f"{sid}_{sc}_r3.nii.gz",
              f"{sid}_{sc}_r2.nii.gz", f"{sid}_{sc}_r1.nii.gz"]:
        fp = os.path.join(sdir, c)
        if os.path.exists(fp):
            mask_path = fp
            break
    if mask_path is None: continue
    
    mask_nib = nib.load(mask_path)
    m2d = (mask_nib.get_fdata()[:, :, sl] > 0.5).astype(np.uint8)
    img_path = find_matching_image(sdir, sid, mask_nib.shape)
    if img_path is None: continue
    img_2d = nib.load(img_path).get_fdata()[:, :, sl].astype(np.float32)
    h, w = img_2d.shape
    
    ys, xs = np.where(m2d > 0)
    gt_box = [max(0, int(xs.min()-5)), max(0, int(ys.min()-5)),
              min(w, int(xs.max()+5)), min(h, int(ys.max()+5))]
    bw, bh = gt_box[2] - gt_box[0], gt_box[3] - gt_box[1]
    disagree = 1.0 - INTER_RATER.get(struct, 0.4)
    std_x, std_y = bw * 0.15 * disagree, bh * 0.15 * disagree
    
    np.random.seed(i)
    noisy_dices = []
    for _ in range(5):
        nb = [max(0, int(gt_box[0] + np.random.normal(0, std_x))),
              max(0, int(gt_box[1] + np.random.normal(0, std_y))),
              min(w, int(gt_box[2] + np.random.normal(0, std_x))),
              min(h, int(gt_box[3] + np.random.normal(0, std_y)))]
        if nb[2] <= nb[0]: nb[2] = nb[0] + 10
        if nb[3] <= nb[1]: nb[3] = nb[1] + 10
        pred = medsam_predict(medsam, img_2d, nb)
        noisy_dices.append(dice(pred, m2d))
    
    entry["dice_simulated"] = round(float(np.mean(noisy_dices)), 4)
    simulated_results.append(entry)
    
    if len(simulated_results) % 50 == 0:
        print(f"  {len(simulated_results)} done...")

print(f"\nSimulated results ({len(simulated_results)} slices):")
for s in ["uterus", "ovary", "endo_cyst"]:
    orc = [r["dice_oracle"] for r in simulated_results if r["structure"] == s]
    sim = [r["dice_simulated"] for r in simulated_results if r["structure"] == s]
    if orc:
        print(f"  {s:12s}: oracle={np.mean(orc):.3f}  sim={np.mean(sim):.3f}")

with open(os.path.join(SAVE_DIR, "simulated_results.json"), "w") as f:
    json.dump(simulated_results, f, indent=2)

## Cell 4: Decoder Fine-Tuning
Freeze encoder (90M) + prompt encoder (6K). Train only decoder (4M).
Pre-compute embeddings for speed.

In [ ]:
# Freeze encoder and prompt encoder
for p in medsam.image_encoder.parameters(): p.requires_grad = False
for p in medsam.prompt_encoder.parameters(): p.requires_grad = False
for p in medsam.mask_decoder.parameters(): p.requires_grad = True
trainable = sum(p.numel() for p in medsam.mask_decoder.parameters() if p.requires_grad)
print(f"Trainable decoder params: {trainable/1e6:.1f}M")

# Collect training slices
print("Collecting training data...")
train_slices = []
for sid in all_subjects:
    if sid not in train_ids: continue
    prefix = sid.split("-")[0]
    sdir = os.path.join(DATASET_ROOT, PREFIX_MAP[prefix], sid)
    if not os.path.isdir(sdir): continue
    for sc in ["ut", "ov", "em", "cy"]:
        mp = None
        for c in [f"{sid}_{sc}.nii.gz", f"{sid}_{sc}_r3.nii.gz",
                  f"{sid}_{sc}_r2.nii.gz", f"{sid}_{sc}_r1.nii.gz",
                  f"{sid}_{sc}_re3.nii.gz"]:
            fp = os.path.join(sdir, c)
            if os.path.exists(fp): mp = fp; break
        if mp is None: continue
        mn = nib.load(mp)
        md = mn.get_fdata()
        if md.max() > 1 or np.unique(md).size > 2: continue
        mask = (md > 0.5).astype(np.uint8)
        ip = find_matching_image(sdir, sid, mn.shape)
        if ip is None: continue
        iv = nib.load(ip).get_fdata().astype(np.float32)
        nsl = min(iv.shape[2], mask.shape[2])
        for sl in range(nsl):
            if mask[:, :, sl].sum() < 10: continue
            if iv[:, :, sl].max() == 0: continue
            train_slices.append({"sid": sid, "img_vol": iv, "mask": mask, "sl": sl})
print(f"Training slices: {len(train_slices)}")

# Pre-compute encoder embeddings
print("Pre-computing embeddings...")
embed_cache = {}
for i, d in enumerate(train_slices):
    key = (d["sid"], d["sl"], id(d["img_vol"]))
    if key in embed_cache: continue
    img_2d = d["img_vol"][:, :, d["sl"]]
    h, w = img_2d.shape
    img_n = norm_pct(img_2d)
    img_3c = np.stack([img_n * 255] * 3, axis=-1).astype(np.uint8)
    img_1024 = np.array(Image.fromarray(img_3c).resize((1024, 1024)))
    img_t = torch.from_numpy(img_1024).permute(2, 0, 1).unsqueeze(0).float().to(device)
    with torch.no_grad():
        emb = medsam.image_encoder(img_t)
    embed_cache[key] = (emb.cpu(), h, w)
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{len(train_slices)}")
print(f"Cached {len(embed_cache)} embeddings")

# Train decoder
optimizer = torch.optim.AdamW(medsam.mask_decoder.parameters(), lr=1e-4, weight_decay=1e-4)
bce = torch.nn.BCELoss()

print("\nTraining decoder...")
best_dice = 0
for epoch in range(1, 11):
    t0 = time.time()
    np.random.shuffle(train_slices)
    epoch_loss, epoch_dice, n = 0, 0, 0
    
    for d in train_slices:
        key = (d["sid"], d["sl"], id(d["img_vol"]))
        if key not in embed_cache: continue
        emb_cpu, h, w = embed_cache[key]
        emb = emb_cpu.to(device)
        m2d = d["mask"][:, :, d["sl"]]
        
        # Oracle bbox
        ys, xs = np.where(m2d > 0)
        bbox = [max(0, int(xs.min()-5)), max(0, int(ys.min()-5)),
                min(w, int(xs.max()+5)), min(h, int(ys.max()+5))]
        sx, sy = 1024/w, 1024/h
        bbox_t = torch.tensor([[bbox[0]*sx, bbox[1]*sy, bbox[2]*sx, bbox[3]*sy]]).float().to(device)
        
        sp, dn = medsam.prompt_encoder(points=None, boxes=bbox_t, masks=None)
        lr, _ = medsam.mask_decoder(image_embeddings=emb,
            image_pe=medsam.prompt_encoder.get_dense_pe(),
            sparse_prompt_embeddings=sp, dense_prompt_embeddings=dn,
            multimask_output=False)
        
        pred = torch.sigmoid(lr)
        pred_256 = F.interpolate(pred, (256, 256), mode="bilinear", align_corners=False)
        gt_256 = torch.from_numpy(
            np.array(Image.fromarray(m2d).resize((256, 256), Image.NEAREST))
        ).float().unsqueeze(0).unsqueeze(0).to(device)
        
        loss_bce = bce(pred_256, gt_256)
        p_flat, g_flat = pred_256.flatten(), gt_256.flatten()
        loss_dice = 1 - (2*(p_flat*g_flat).sum()+1)/(p_flat.sum()+g_flat.sum()+1)
        loss = loss_bce + loss_dice
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(medsam.mask_decoder.parameters(), 1.0)
        optimizer.step()
        
        epoch_loss += loss.item()
        with torch.no_grad():
            d_val = (2*((pred_256>0.5).float()*gt_256).sum()+1)/((pred_256>0.5).float().sum()+gt_256.sum()+1)
            epoch_dice += d_val.item()
        n += 1
    
    avg_loss = epoch_loss / max(n, 1)
    avg_dice = epoch_dice / max(n, 1)
    marker = ""
    if avg_dice > best_dice:
        best_dice = avg_dice
        torch.save({"decoder": medsam.mask_decoder.state_dict(), "epoch": epoch},
                   os.path.join(SAVE_DIR, "best_decoder_big.pth"))
        marker = " <- BEST"
    print(f"  Epoch {epoch:2d} | Loss {avg_loss:.4f} | Dice {avg_dice:.4f} | {time.time()-t0:.0f}s{marker}")

print(f"Best training Dice: {best_dice:.4f}")

## Cell 5: Decoder Test Evaluation (234 slices)

In [ ]:
# Load best decoder
dec = torch.load(os.path.join(SAVE_DIR, "best_decoder_big.pth"),
                 map_location="cpu", weights_only=False)
medsam.mask_decoder.load_state_dict(dec["decoder"])
medsam.eval()

decoder_results = []
for sid in test_ids:
    prefix = sid.split("-")[0]
    sdir = os.path.join(DATASET_ROOT, PREFIX_MAP[prefix], sid)
    if not os.path.isdir(sdir): continue
    for sc in ["ut", "ov", "em", "cy"]:
        mp = None
        for c in [f"{sid}_{sc}.nii.gz", f"{sid}_{sc}_r3.nii.gz",
                  f"{sid}_{sc}_r2.nii.gz", f"{sid}_{sc}_r1.nii.gz",
                  f"{sid}_{sc}_re3.nii.gz"]:
            fp = os.path.join(sdir, c)
            if os.path.exists(fp): mp = fp; break
        if mp is None: continue
        mn = nib.load(mp); md = mn.get_fdata()
        if md.max() > 1 or np.unique(md).size > 2: continue
        gt = (md > 0.5).astype(np.uint8)
        ip = find_matching_image(sdir, sid, mn.shape)
        if ip is None: continue
        iv = nib.load(ip).get_fdata().astype(np.float32)
        nsl = min(iv.shape[2], gt.shape[2])
        for sl in range(nsl):
            m2d = gt[:, :, sl]
            if m2d.sum() < 10 or iv[:, :, sl].max() == 0: continue
            ys, xs = np.where(m2d > 0)
            h, w = iv.shape[:2]
            bbox = [max(0, int(xs.min()-5)), max(0, int(ys.min()-5)),
                    min(w, int(xs.max()+5)), min(h, int(ys.max()+5))]
            pred = medsam_predict(medsam, iv[:, :, sl], bbox)
            d = dice(pred, m2d)
            decoder_results.append({"struct": sc, "dice": round(d, 4)})

print(f"Decoder FT test ({len(decoder_results)} slices):")
for s in ["ut", "ov", "em", "cy"]:
    e = [r["dice"] for r in decoder_results if r["struct"] == s]
    if e:
        print(f"  {s}: {np.mean(e):.3f} +/- {np.std(e):.3f} ({len(e)} slices)")

with open(os.path.join(SAVE_DIR, "decoder_test_big.json"), "w") as f:
    json.dump(decoder_results, f, indent=2)

## Cell 6: Auto-Box Pipeline (U-Net -> MedSAM and Binary -> MedSAM)
Uses U-Net or binary model predictions as bounding box prompts for MedSAM

In [ ]:
import segmentation_models_pytorch as smp

UNET_PATH = os.path.expanduser("~/Desktop/EndoSAM_UNet_Compare/vanilla_unet_best.pth")
BIN_OV_PATH = os.path.expanduser("~/Desktop/EndoSAM_BinaryOvary/best_binary_ovary.pth")
BIN_EN_PATH = os.path.expanduser("~/Desktop/EndoSAM_BinaryEndo/best_binary_endo.pth")

def load_unet_for_bbox(path, classes=4, attention=None):
    m = smp.Unet(encoder_name="efficientnet-b0", encoder_weights=None,
                 in_channels=3, classes=classes,
                 decoder_attention_type=attention).to(device)
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    m.load_state_dict(ckpt["model"], strict=False)
    m.eval()
    return m

def predict_bbox(model, img_vol, sl, n_sl, h, w, target_class):
    """Get prediction from U-Net/binary and extract bounding box"""
    chs = [norm_pct(img_vol[:, :, min(max(0, sl+o), n_sl-1)]) for o in [-1, 0, 1]]
    img_3 = np.stack(chs, axis=-1)
    img_r = np.array(Image.fromarray((img_3*255).astype(np.uint8)).resize((256, 256)))/255.0
    img_t = torch.from_numpy(img_r).permute(2, 0, 1).float().unsqueeze(0)
    with torch.no_grad():
        pred = model(img_t.to(device)).argmax(dim=1)[0].cpu().numpy()
    pred_orig = np.array(Image.fromarray(pred.astype(np.uint8)).resize((w, h), Image.NEAREST))
    mask_pred = (pred_orig == target_class).astype(np.uint8)
    if mask_pred.sum() < 5:
        return None
    ys, xs = np.where(mask_pred > 0)
    return [max(0, int(xs.min()-10)), max(0, int(ys.min()-10)),
            min(w, int(xs.max()+10)), min(h, int(ys.max()+10))]

# Test both pipelines
STRUCT_CH = {"ut": 1, "ov": 2, "em": 3, "cy": 3}

for pipeline_name, box_configs in [
    ("Multi-class U-Net", {"model_path": UNET_PATH, "classes": 4, "structs": ["ut", "ov", "em", "cy"]}),
]:
    if not os.path.exists(box_configs["model_path"]):
        print(f"SKIP {pipeline_name}: model not found")
        continue
    
    box_model = load_unet_for_bbox(box_configs["model_path"], box_configs["classes"])
    results = []
    
    for sid in test_ids:
        prefix = sid.split("-")[0]
        sdir = os.path.join(DATASET_ROOT, PREFIX_MAP[prefix], sid)
        if not os.path.isdir(sdir): continue
        for sc in box_configs["structs"]:
            mp = None
            for c in [f"{sid}_{sc}.nii.gz", f"{sid}_{sc}_r3.nii.gz",
                      f"{sid}_{sc}_r2.nii.gz", f"{sid}_{sc}_r1.nii.gz"]:
                fp = os.path.join(sdir, c)
                if os.path.exists(fp): mp = fp; break
            if mp is None: continue
            mn = nib.load(mp); md = mn.get_fdata()
            if md.max() > 1 or np.unique(md).size > 2: continue
            gt = (md > 0.5).astype(np.uint8)
            ip = find_matching_image(sdir, sid, mn.shape)
            if ip is None: continue
            iv = nib.load(ip).get_fdata().astype(np.float32)
            nsl = min(iv.shape[2], gt.shape[2])
            h, w = iv.shape[:2]
            for sl in range(nsl):
                m2d = gt[:, :, sl]
                if m2d.sum() < 10 or iv[:, :, sl].max() == 0: continue
                bbox = predict_bbox(box_model, iv, sl, nsl, h, w, STRUCT_CH[sc])
                if bbox is None:
                    results.append({"struct": sc, "dice": 0.0, "found": False})
                    continue
                pred = medsam_predict(medsam, iv[:, :, sl], bbox)
                d = dice(pred, m2d)
                results.append({"struct": sc, "dice": round(d, 4), "found": True})
    
    print(f"\n{pipeline_name} auto-box:")
    for s in ["ut", "ov", "em", "cy"]:
        e = [r for r in results if r["struct"] == s]
        found = [r for r in e if r["found"]]
        if e:
            print(f"  {s}: {np.mean([r['dice'] for r in found]):.3f} "
                  f"(found {len(found)}/{len(e)})")

# Binary auto-box (ovary + endo)
print("\nBinary auto-box:")
for struct, path, tc in [("ov", BIN_OV_PATH, 1), ("em", BIN_EN_PATH, 1), ("cy", BIN_EN_PATH, 1)]:
    if not os.path.exists(path):
        print(f"  SKIP {struct}: model not found")
        continue
    box_model = load_unet_for_bbox(path, 2)
    results = []
    for sid in test_ids:
        prefix = sid.split("-")[0]
        sdir = os.path.join(DATASET_ROOT, PREFIX_MAP[prefix], sid)
        if not os.path.isdir(sdir): continue
        mp = None
        for c in [f"{sid}_{struct}.nii.gz", f"{sid}_{struct}_r3.nii.gz",
                  f"{sid}_{struct}_r2.nii.gz"]:
            fp = os.path.join(sdir, c)
            if os.path.exists(fp): mp = fp; break
        if mp is None: continue
        mn = nib.load(mp); md = mn.get_fdata()
        if md.max() > 1 or np.unique(md).size > 2: continue
        gt = (md > 0.5).astype(np.uint8)
        ip = find_matching_image(sdir, sid, mn.shape)
        if ip is None: continue
        iv = nib.load(ip).get_fdata().astype(np.float32)
        nsl = min(iv.shape[2], gt.shape[2])
        h, w = iv.shape[:2]
        for sl in range(nsl):
            m2d = gt[:, :, sl]
            if m2d.sum() < 10 or iv[:, :, sl].max() == 0: continue
            bbox = predict_bbox(box_model, iv, sl, nsl, h, w, tc)
            if bbox is None:
                results.append({"dice": 0.0, "found": False})
                continue
            pred = medsam_predict(medsam, iv[:, :, sl], bbox)
            d = dice(pred, m2d)
            results.append({"dice": round(d, 4), "found": True})
    
    found = [r for r in results if r["found"]]
    if results:
        print(f"  {struct}: {np.mean([r['dice'] for r in found]):.3f} "
              f"(found {len(found)}/{len(results)})")

print("\nAll MedSAM experiments complete.")